# IAN 601 — Module 10: Pandas for Real Analytics

Pandas is one of the most widely used Python libraries for working with tabular data.

A Pandas **DataFrame** represents rows and columns similar to a spreadsheet or database table, while still allowing us to use Python for filtering, cleaning, transformation, aggregation, and validation.

This notebook is designed to stand alone and uses one sales dataset throughout.

**Workflow:** load → inspect → select → clean → transform → summarize → combine → validate

## Learning goals

By the end of this notebook, you should be able to:

1. load CSV data into a DataFrame;
2. inspect rows, columns, shape, and data types;
3. select columns with bracket notation;
4. select rows and columns with `loc` and `iloc`;
5. filter and sort records;
6. identify missing values and duplicates;
7. convert text, numeric, and date columns safely;
8. create derived columns;
9. summarize data with `groupby()` and `agg()`;
10. create pivot tables;
11. combine tables with `merge()` and `concat()`;
12. validate the cleaned result before reporting it.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

### Expandable syntax guides

Each major section has a collapsed **Syntax & parameters** box. Open it when you want function signatures, important parameters, related methods, or usage cautions. Keep it closed when you want to focus on the main workflow.

## 1. Import Pandas

Pandas is usually imported with the alias `pd`.

<details>
<summary><strong>Syntax & parameters — importing Pandas</strong></summary>

<pre><code class="language-python">import pandas as pd</code></pre>

<p><code>pd</code> is the conventional alias used in documentation and most analytics code.</p>

<p><strong>Useful display options</strong></p>
<pre><code class="language-python">pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)</code></pre>

<p>Use <code>pd.get_option()</code> to inspect an option and <code>pd.reset_option()</code> to restore defaults.</p>

<p><strong>Related objects</strong></p>
<ul>
<li><code>pd.Series</code> — one-dimensional labeled data.</li>
<li><code>pd.DataFrame</code> — two-dimensional labeled table.</li>
<li><code>pd.Index</code> — labels used for rows or columns.</li>
</ul>
</details>

In [ ]:
import pandas as pd

pd.__version__

## 2. Load the Module 10 dataset

When you open this notebook in Colab, read the CSV directly from GitHub.

<details>
<summary><strong>Syntax & parameters — read_csv()</strong></summary>

<pre><code class="language-python">pd.read_csv(
    "file.csv",
    sep=",",
    usecols=None,
    dtype=None,
    parse_dates=None,
    na_values=None
)</code></pre>

<p><strong>Common parameters</strong></p>
<ul>
<li><code>sep</code> — delimiter such as comma, tab, or pipe.</li>
<li><code>usecols</code> — load selected columns only.</li>
<li><code>dtype</code> — force column types.</li>
<li><code>parse_dates</code> — parse selected columns as datetime.</li>
<li><code>na_values</code> — extra strings to treat as missing.</li>
<li><code>nrows</code> — load only the first N rows.</li>
<li><code>encoding</code> — text encoding, commonly <code>"utf-8"</code>.</li>
</ul>

<p>Related readers include <code>read_excel()</code>, <code>read_json()</code>, <code>read_sql()</code>, and <code>read_parquet()</code>.</p>
</details>

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module10_Module_Sales_300.csv"

orders = pd.read_csv(data_url)
orders.head()

If you download the notebook and CSV to the same folder on your computer, you can instead use:

```python
orders = pd.read_csv("IAN601_Module10_Module_Sales_300.csv")
```

## 3. First inspection

Before cleaning anything, check what actually loaded.

<details>
<summary><strong>Syntax & parameters — inspecting a DataFrame</strong></summary>

<pre><code class="language-python">df.head(5)
df.tail(5)
df.sample(5, random_state=1)
df.shape
df.columns
df.index
df.dtypes
df.info()</code></pre>

<ul>
<li><code>head(n)</code> returns the first N rows.</li>
<li><code>tail(n)</code> returns the last N rows.</li>
<li><code>sample(n)</code> returns random rows.</li>
<li><code>shape</code> is a tuple: <code>(rows, columns)</code>.</li>
<li><code>info()</code> shows non-null counts, dtypes, and memory usage.</li>
</ul>

<p>For a quick numeric summary, use <code>df.describe()</code>. For categorical columns too, try <code>df.describe(include="all")</code>.</p>
</details>

In [ ]:
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())

orders.head(8)

In [ ]:
orders.sample(5, random_state=10)

## 4. Data types and structure

The `amount` and `order_date` columns are intentionally not fully clean yet.

<details>
<summary><strong>Syntax & parameters — dtypes and conversion</strong></summary>

<pre><code class="language-python">df.dtypes
df["quantity"] = df["quantity"].astype("int64")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.convert_dtypes()</code></pre>

<p><strong>Common Pandas dtypes</strong></p>
<ul>
<li><code>int64</code> — integers.</li>
<li><code>float64</code> — decimal numbers.</li>
<li><code>object</code> — often text or mixed values.</li>
<li><code>string</code> — Pandas string dtype.</li>
<li><code>boolean</code> — nullable Boolean dtype.</li>
<li><code>datetime64[ns]</code> — datetime values.</li>
<li><code>category</code> — repeated categorical labels.</li>
</ul>

<p><code>astype()</code> expects conversion to succeed. <code>to_numeric()</code> and <code>to_datetime()</code> support <code>errors="coerce"</code>, which converts invalid values to missing values.</p>
</details>

In [ ]:
orders.dtypes

In [ ]:
orders.info()

## 5. Select columns

A single column returns a Series. A list of columns returns a DataFrame.

<details>
<summary><strong>Syntax & parameters — selecting columns</strong></summary>

<pre><code class="language-python">df["amount"]                  # Series
df[["order_id", "amount"]]    # DataFrame</code></pre>

<p>A single column name returns a <code>Series</code>. A list of column names returns a <code>DataFrame</code>.</p>

<p><strong>Other useful selectors</strong></p>
<pre><code class="language-python">df.filter(items=["order_id", "amount"])
df.filter(like="sales")
df.select_dtypes(include="number")
df.select_dtypes(exclude="object")</code></pre>

<p>Use a list when you want the result to remain two-dimensional.</p>
</details>

In [ ]:
orders["amount"].head()

In [ ]:
orders[["order_id", "channel", "amount"]].head()

## 6. Use loc and iloc

- `loc` selects by labels or conditions.
- `iloc` selects by integer position.

<details>
<summary><strong>Syntax & parameters — loc and iloc</strong></summary>

<p><strong>Label-based selection</strong></p>
<pre><code class="language-python">df.loc[row_labels, column_labels]</code></pre>

<p><strong>Position-based selection</strong></p>
<pre><code class="language-python">df.iloc[row_positions, column_positions]</code></pre>

<p>Examples:</p>
<pre><code class="language-python">df.loc[0:4, ["order_id", "amount"]]
df.loc[df["amount"] &gt; 100, ["order_id", "amount"]]
df.iloc[0:5, 0:3]</code></pre>

<p><code>loc</code> label slices are inclusive of the ending label. <code>iloc</code> follows normal Python slicing and excludes the ending position.</p>

<p>For one scalar value, <code>at[]</code> and <code>iat[]</code> are lightweight alternatives.</p>
</details>

In [ ]:
orders.loc[0:4, ["order_id", "channel", "amount"]]

In [ ]:
orders.iloc[0:5, 0:4]

## 7. Filter and sort rows

For now, filter using columns that are already numeric or clean enough to use.

<details>
<summary><strong>Syntax & parameters — filtering and sorting</strong></summary>

<pre><code class="language-python">df[df["amount"] &gt;= 100]

df[
    (df["amount"] &gt;= 100)
    &amp; (df["status"] == "complete")
]</code></pre>

<p>With Boolean indexing, wrap each condition in parentheses and use <code>&amp;</code>, <code>|</code>, and <code>~</code>.</p>

<p><strong>Useful filtering helpers</strong></p>
<pre><code class="language-python">df["status"].isin(["complete", "returned"])
df["amount"].between(50, 100)
df["name"].str.contains("abc", case=False, na=False)</code></pre>

<p><strong>Sorting</strong></p>
<pre><code class="language-python">df.sort_values(
    by=["region", "amount"],
    ascending=[True, False],
    na_position="last"
)</code></pre>

<p>Related methods: <code>nlargest()</code>, <code>nsmallest()</code>, and <code>sort_index()</code>.</p>
</details>

In [ ]:
web_orders = orders[orders["channel"].str.strip().str.lower() == "web"]

web_orders[["order_id", "channel", "quantity"]].head()

In [ ]:
orders.sort_values("quantity", ascending=False).head(10)

## 8. Profile data quality

Check missing values and duplicate rows before making changes.

<details>
<summary><strong>Syntax & parameters — missing values and duplicates</strong></summary>

<pre><code class="language-python">df.isna().sum()
df.isna().mean() * 100
df.notna().sum()

df.duplicated().sum()
df.duplicated(subset=["order_id"], keep=False)</code></pre>

<p><strong>duplicated() parameters</strong></p>
<ul>
<li><code>subset</code> — columns used to define a duplicate.</li>
<li><code>keep="first"</code> — mark later duplicates.</li>
<li><code>keep="last"</code> — mark earlier duplicates.</li>
<li><code>keep=False</code> — mark every member of a duplicate group.</li>
</ul>

<p>Useful profiling methods also include <code>value_counts()</code>, <code>nunique()</code>, <code>describe()</code>, <code>min()</code>, and <code>max()</code>.</p>
</details>

In [ ]:
orders.isna().sum()

In [ ]:
print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
orders[orders.duplicated(keep=False)].sort_values("order_id")

## 9. Create a clean working copy

Keep the original DataFrame available so you can compare before and after.

<details>
<summary><strong>Syntax & parameters — copying and common cleaning tools</strong></summary>

<pre><code class="language-python">clean = df.copy()</code></pre>

<p>Using <code>copy()</code> keeps the original DataFrame available for comparison.</p>

<p><strong>Common cleaning methods</strong></p>
<pre><code class="language-python">df["text"].str.strip()
df["text"].str.lower()
df["text"].str.replace("$", "", regex=False)

pd.to_numeric(df["amount"], errors="coerce")
pd.to_datetime(df["date"], errors="coerce")

df["city"].fillna("Unknown")
df["amount"].fillna(df["amount"].median())

df.drop_duplicates()
df.drop_duplicates(subset="order_id")</code></pre>

<p>Related methods: <code>dropna()</code>, <code>replace()</code>, <code>interpolate()</code>, and <code>convert_dtypes()</code>.</p>

<p>Cleaning decisions should be based on meaning, not only convenience.</p>
</details>

In [ ]:
clean = orders.copy()

### Clean text columns

In [ ]:
clean["channel"] = clean["channel"].str.strip().str.lower()
clean["status"] = clean["status"].str.strip().str.lower()

clean[["channel", "status"]].head(10)

### Convert amount to numeric

The CSV stores amounts with a dollar sign. Remove it first, then convert.

In [ ]:
clean["amount"] = pd.to_numeric(
    clean["amount"].str.replace("$", "", regex=False),
    errors="coerce"
)

clean["amount"].head(10)

### Convert dates safely

Invalid dates become missing date values when `errors="coerce"` is used.

In [ ]:
clean["order_date"] = pd.to_datetime(
    clean["order_date"],
    errors="coerce"
)

clean[["order_id", "order_date"]].head()

### Handle missing values

The choice depends on the meaning of the column. Here we use simple teaching examples.

In [ ]:
clean["city"] = clean["city"].fillna("Unknown")
clean["region"] = clean["region"].fillna("Unknown")
clean["amount"] = clean["amount"].fillna(clean["amount"].median())

clean.isna().sum()

### Remove exact duplicate rows

In [ ]:
print("Rows before:", len(clean))

clean = clean.drop_duplicates().copy()

print("Rows after:", len(clean))

## 10. Create derived columns

<details>
<summary><strong>Syntax & parameters — derived columns</strong></summary>

<pre><code class="language-python">df["tax"] = df["amount"] * 0.07
df["total"] = df["amount"] + df["tax"]</code></pre>

<p>Vectorized column arithmetic is usually preferable to row-by-row loops.</p>

<p><strong>assign() alternative</strong></p>
<pre><code class="language-python">df = df.assign(
    tax=lambda x: x["amount"] * 0.07,
    total=lambda x: x["amount"] + x["tax"]
)</code></pre>

<p><strong>Conditional columns</strong></p>
<pre><code class="language-python">import numpy as np

df["large_order"] = np.where(
    df["amount"] &gt;= 100,
    "yes",
    "no"
)</code></pre>

<p>Related tools: <code>where()</code>, <code>mask()</code>, <code>map()</code>, <code>replace()</code>, and <code>cut()</code>.</p>
</details>

In [ ]:
clean["tax"] = clean["amount"] * 0.07
clean["total_with_tax"] = clean["amount"] + clean["tax"]

clean[["order_id", "amount", "tax", "total_with_tax"]].head().round(2)

In [ ]:
clean["month"] = clean["order_date"].dt.month_name()
clean["weekday"] = clean["order_date"].dt.day_name()
clean["quarter"] = clean["order_date"].dt.quarter

clean[["order_date", "month", "weekday", "quarter"]].head()

## 11. GroupBy and aggregation

Now that the columns are cleaner, answer questions by category.

<details>
<summary><strong>Syntax & parameters — groupby() and agg()</strong></summary>

<pre><code class="language-python">df.groupby("channel", as_index=False).agg(
    orders=("order_id", "count"),
    revenue=("amount", "sum"),
    avg_order=("amount", "mean")
)</code></pre>

<p><strong>Common groupby() parameters</strong></p>
<ul>
<li><code>by</code> — one or more grouping columns.</li>
<li><code>as_index=False</code> — keep group keys as ordinary columns.</li>
<li><code>dropna=False</code> — keep missing labels as a group.</li>
<li><code>sort=False</code> — preserve observed group order when appropriate.</li>
</ul>

<p><strong>Common aggregations</strong>: <code>sum</code>, <code>mean</code>, <code>median</code>, <code>count</code>, <code>size</code>, <code>min</code>, <code>max</code>, <code>std</code>, <code>nunique</code>.</p>

<p><code>size()</code> counts rows. <code>count()</code> counts non-missing values in a selected column.</p>
</details>

In [ ]:
channel_summary = (
    clean.groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum"),
        avg_order=("amount", "mean")
    )
)

channel_summary.round(2)

In [ ]:
region_summary = (
    clean.groupby("region", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum")
    )
    .sort_values("revenue", ascending=False)
)

region_summary.round(2)

## 12. Pivot table

A pivot table summarizes values across two categorical dimensions.

<details>
<summary><strong>Syntax & parameters — pivot_table()</strong></summary>

<pre><code class="language-python">pd.pivot_table(
    df,
    index="region",
    columns="channel",
    values="amount",
    aggfunc="sum",
    fill_value=0
)</code></pre>

<ul>
<li><code>index</code> — categories placed on rows.</li>
<li><code>columns</code> — categories placed across columns.</li>
<li><code>values</code> — numeric values to summarize.</li>
<li><code>aggfunc</code> — summary function; can also be a list.</li>
<li><code>fill_value</code> — replace empty cells in the result.</li>
<li><code>margins=True</code> — add totals.</li>
</ul>

<p><code>pivot()</code> reshapes without aggregation. <code>pivot_table()</code> can aggregate duplicate row/column combinations.</p>
</details>

In [ ]:
pivot = pd.pivot_table(
    clean,
    values="amount",
    index="region",
    columns="channel",
    aggfunc="sum",
    fill_value=0
)

pivot.round(2)

## 13. Merge related tables

Create a small customer lookup table and merge it with the order data.

<details>
<summary><strong>Syntax & parameters — merge()</strong></summary>

<pre><code class="language-python">left.merge(
    right,
    on="customer_id",
    how="left",
    validate="many_to_one"
)</code></pre>

<p><strong>Join types</strong>: <code>left</code>, <code>inner</code>, <code>right</code>, <code>outer</code>.</p>

<p><strong>Useful parameters</strong></p>
<ul>
<li><code>on</code> — shared key name.</li>
<li><code>left_on</code>, <code>right_on</code> — different key names.</li>
<li><code>suffixes</code> — labels for overlapping non-key columns.</li>
<li><code>validate</code> — verify one-to-one, one-to-many, many-to-one, or many-to-many structure.</li>
<li><code>indicator=True</code> — add a <code>_merge</code> column to inspect matches.</li>
</ul>

<p>Always compare row counts and inspect unmatched keys after important joins.</p>
</details>

In [ ]:
customer_ids = sorted(clean["customer_id"].dropna().unique())

customers = pd.DataFrame({
    "customer_id": customer_ids,
    "segment": [
        ["consumer", "business", "education"][i % 3]
        for i in range(len(customer_ids))
    ]
})

customers.head()

In [ ]:
merged = clean.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

merged[["order_id", "customer_id", "segment", "amount"]].head()

## 14. Concatenate similar tables

Concatenation stacks tables with similar columns.

<details>
<summary><strong>Syntax & parameters — concat()</strong></summary>

<pre><code class="language-python">pd.concat(
    [df1, df2],
    axis=0,
    ignore_index=True
)</code></pre>

<ul>
<li><code>axis=0</code> — stack rows.</li>
<li><code>axis=1</code> — place objects side by side as columns.</li>
<li><code>ignore_index=True</code> — create a fresh row index.</li>
<li><code>join="outer"</code> — keep all columns.</li>
<li><code>join="inner"</code> — keep common columns only.</li>
<li><code>keys</code> — label the source of each concatenated object.</li>
</ul>

<p><code>concat()</code> combines by position/labels. <code>merge()</code> combines by key relationships.</p>
</details>

In [ ]:
january = clean[clean["order_date"].dt.month == 1].head(5)
february = clean[clean["order_date"].dt.month == 2].head(5)

sample_two_months = pd.concat(
    [january, february],
    ignore_index=True
)

sample_two_months[["order_id", "order_date", "channel", "amount"]]

## 15. Validate the cleaned data

A workflow is not finished until you check the result.

<details>
<summary><strong>Syntax & parameters — validation checks</strong></summary>

<pre><code class="language-python">len(df)
df.shape
df.isna().sum()
df.duplicated().sum()
df["order_id"].nunique()
df["amount"].sum()</code></pre>

<p><strong>Assertions</strong></p>
<pre><code class="language-python">assert df["order_id"].is_unique
assert df["amount"].notna().all()
assert (df["amount"] &gt;= 0).all()</code></pre>

<p><strong>Useful checks</strong></p>
<ul>
<li><code>is_unique</code> — whether a Series has unique values.</li>
<li><code>between()</code> — range checks.</li>
<li><code>isin()</code> — valid-category checks.</li>
<li><code>equals()</code> — exact object comparison.</li>
</ul>

<p>Validation should reflect the meaning of the data, not only technical properties.</p>
</details>

In [ ]:
print("Rows:", len(clean))
print("Duplicate rows:", clean.duplicated().sum())
print("Duplicate order IDs:", clean["order_id"].duplicated().sum())
print("Missing amount values:", clean["amount"].isna().sum())
print("Invalid/missing dates:", clean["order_date"].isna().sum())
print("Total revenue:", round(clean["amount"].sum(), 2))

## 16. Compact analyst workflow

The steps below are the main pattern to remember:

1. Load
2. Inspect
3. Clean
4. Transform
5. Summarize
6. Validate

<details>
<summary><strong>Syntax guide — building a compact Pandas workflow</strong></summary>

<pre><code class="language-python">result = (
    df
    .query(...)
    .assign(...)
    .groupby(..., as_index=False)
    .agg(...)
    .sort_values(...)
)</code></pre>

<p>A readable chain usually follows analytical stages:</p>
<ol>
<li>filter;</li>
<li>create or clean columns;</li>
<li>group;</li>
<li>aggregate;</li>
<li>sort or format the result.</li>
</ol>

<p>For longer workflows, use <code>pipe()</code> with named helper functions and inspect intermediate results while developing the analysis.</p>
</details>

In [ ]:
analysis = (
    clean[clean["status"] == "complete"]
    .groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "count"),
        revenue=("amount", "sum"),
        avg_order=("amount", "mean")
    )
    .sort_values("revenue", ascending=False)
)

analysis.round(2)

## Practice

Try these without copying an earlier cell:

1. Find the five largest completed orders.
2. Calculate total revenue by category.
3. Create a pivot table with category as rows and channel as columns.
4. Find the number of orders in each month.
5. Check whether every customer ID in the order table matched a customer segment.

<details>
<summary><strong>Hints & syntax — practice section</strong></summary>

<p>Methods that may help:</p>
<pre><code class="language-python">df.query(...)
df.nlargest(...)
df.groupby(...).agg(...)
pd.pivot_table(...)
df.merge(...)
df.isna().sum()</code></pre>

<p>Try to solve each question first. Open earlier sections only when you need a reminder of the pattern.</p>
</details>

In [ ]:
# 1. Five largest completed orders


# 2. Revenue by category


# 3. Category x channel pivot table


# 4. Orders by month


# 5. Check the merge


## Summary and self-check

A reliable Pandas workflow is more important than memorizing dozens of methods.

Make sure you can explain:

- the difference between a Series and a DataFrame;
- what `head()`, `shape`, `dtypes`, and `info()` tell you;
- when to use `loc` versus `iloc`;
- why type conversion should be checked carefully;
- what `groupby()` does;
- how a pivot table summarizes categories;
- the difference between `merge()` and `concat()`;
- why row counts, missingness, duplicates, and totals should be checked after cleaning.

Use the practice section to rebuild part of the workflow without copying earlier cells.